# Exercise 5 — Full Pipeline: Strategy + Risk + Paper Trader

The complete Section 7 stack, end to end: generate signals with an SMA crossover, apply risk filters (stop-loss + drawdown limit), then run the paper-trading bot. Compare raw vs. risk-filtered results to see what risk management costs and what it buys.

In [ ]:
import pandas as pd, math
from dataclasses import dataclass, field

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
@dataclass
class Trade:
    date:        object
    action:      str
    price:       float
    shares:      float
    cash_after:  float
    value_after: float
@dataclass
class PaperAccount:
    initial_cash: float = 10_000.0
    cash:         float = field(init=False)
    shares:       float = field(init=False)
    trades:       list  = field(init=False)

    def __post_init__(self):
        self.cash   = self.initial_cash
        self.shares = 0.0
        self.trades = []

    def portfolio_value(self, price):
        return self.cash + self.shares * float(price)

    def buy(self, date, price, fraction=1.0):
        price = float(price)
        if self.cash <= 0 or price <= 0:
            return None
        shares = (self.cash * fraction) / price
        cost   = shares * price
        if cost > self.cash:
            shares = self.cash / price
            cost   = shares * price
        self.cash   -= cost
        self.shares += shares
        t = Trade(date=date, action="BUY", price=price, shares=shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t

    def sell(self, date, price):
        price = float(price)
        if self.shares <= 0:
            return None
        proceeds    = self.shares * price
        sold_shares = self.shares
        self.cash  += proceeds
        self.shares = 0.0
        t = Trade(date=date, action="SELL", price=price, shares=sold_shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t
def run_paper_trader(df, signals, initial_cash=10_000.0, fraction=1.0):
    account = PaperAccount(initial_cash=initial_cash)
    eq_values   = []
    prev_signal = 0
    for i in range(len(df)):
        date  = df.index[i]
        price = float(df["Close"].iloc[i])
        sig   = int(signals.iloc[i])
        if sig == 1 and prev_signal == 0:
            account.buy(date, price, fraction=fraction)
        elif sig == 0 and prev_signal == 1:
            account.sell(date, price)
        eq_values.append(account.portfolio_value(price))
        prev_signal = sig
    if account.shares > 0:
        account.sell(df.index[-1], float(df["Close"].iloc[-1]))
    equity       = pd.Series(eq_values, index=df.index)
    total_return = float(equity.iloc[-1] / initial_cash - 1.0)
    peak         = equity.cummax()
    max_dd       = float(((equity - peak) / peak).min())
    return {
        "account":      account,
        "trades":       account.trades,
        "equity":       equity,
        "initial_cash": initial_cash,
        "final_value":  float(equity.iloc[-1]),
        "total_return": total_return,
        "max_drawdown": max_dd,
        "n_trades":     len(account.trades),
        "n_buys":       sum(1 for t in account.trades if t.action == "BUY"),
        "n_sells":      sum(1 for t in account.trades if t.action == "SELL"),
    }
def format_report(result):
    lines = [
        "=== Paper Trading Report ===",
        f"Initial cash :  ${result['initial_cash']:>12,.2f}",
        f"Final value  :  ${result['final_value']:>12,.2f}",
        f"Total return :  {result['total_return']:>12.2%}",
        f"Max drawdown :  {result['max_drawdown']:>12.2%}",
        f"Trades total :  {result['n_trades']:>12d}",
        f"  Buys       :  {result['n_buys']:>12d}",
        f"  Sells      :  {result['n_sells']:>12d}",
    ]
    if result["trades"]:
        first = result["trades"][0]
        last  = result["trades"][-1]
        lines.append(f"First trade  :  {first.action} @ ${first.price:,.2f}  ({first.date})")
        lines.append(f"Last trade   :  {last.action} @ ${last.price:,.2f}  ({last.date})")
    return "\n".join(lines)
def _sma_cross(df, fast=20, slow=50):
    c = df["Close"]
    return (c.rolling(fast).mean() > c.rolling(slow).mean()).fillna(False).astype(int)

def _apply_sl(signals, prices, stop_pct=0.05):
    result = signals.copy().astype(float)
    entry  = None
    for i in range(len(result)):
        if result.iloc[i] == 1:
            if entry is None:
                entry = float(prices.iloc[i])
            elif float(prices.iloc[i]) <= entry * (1 - stop_pct):
                result.iloc[i] = 0
                entry = None
        else:
            entry = None
    return result.astype(int)

def _apply_dd(signals, prices, limit=-0.20):
    peak = prices.cummax()
    dd   = (prices - peak) / peak
    r    = signals.copy().astype(int)
    r[dd < limit] = 0
    return r


In [ ]:
df = _synthetic(n=252)

# Generate raw signals
raw_signals = _sma_cross(df)

# Apply risk filters
sl_signals  = _apply_sl(raw_signals, df["Close"], stop_pct=0.05)
safe_signals = _apply_dd(sl_signals, df["Close"], limit=-0.20)

# Run paper traders
r_raw  = run_paper_trader(df, raw_signals,  initial_cash=10_000.0)
r_safe = run_paper_trader(df, safe_signals, initial_cash=10_000.0)

print("--- RAW STRATEGY ---")
print(format_report(r_raw))
print()
print("--- RISK-FILTERED ---")
print(format_report(r_safe))


### Checks

In [ ]:
checks = 0

# 1 — risk-filtered max_drawdown ≥ raw (less severe)
try:
    assert r_safe["max_drawdown"] >= r_raw["max_drawdown"],         f"risk max_dd ({r_safe['max_drawdown']:.2%}) should be ≥ raw ({r_raw['max_drawdown']:.2%})"
    checks += 1; print("✅ 1 risk-filtered max_drawdown is less severe (or equal)")
except Exception as e:
    print("❌ 1:", e)

# 2 — risk-filtered has ≤ raw n_trades
try:
    assert r_safe["n_trades"] <= r_raw["n_trades"],         f"risk {r_safe['n_trades']} trades should be ≤ raw {r_raw['n_trades']}"
    checks += 1; print("✅ 2 risk-filtered has fewer or equal trades than raw")
except Exception as e:
    print("❌ 2:", e)

# 3 — equity length and index match df
try:
    for label, r in [("raw", r_raw), ("safe", r_safe)]:
        assert len(r["equity"]) == len(df)
        assert (r["equity"].index == df.index).all()
    checks += 1; print("✅ 3 equity Series length and index correct for both")
except Exception as e:
    print("❌ 3:", e)

# 4 — final_value = equity.iloc[-1]
try:
    for label, r in [("raw", r_raw), ("safe", r_safe)]:
        assert abs(r["final_value"] - r["equity"].iloc[-1]) < 1e-9,             f"{label}: final_value != equity[-1]"
    checks += 1; print("✅ 4 final_value == equity.iloc[-1] for both strategies")
except Exception as e:
    print("❌ 4:", e)

# 5 — n_buys == n_sells (or n_sells == n_buys - 1 if force-sold)
try:
    for label, r in [("raw", r_raw), ("safe", r_safe)]:
        diff = abs(r["n_buys"] - r["n_sells"])
        assert diff <= 1, f"{label}: |buys-sells| should be 0 or 1, got {diff}"
    checks += 1; print("✅ 5 |n_buys - n_sells| ≤ 1 for both (forced sell closes last trade)")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
